# L5a: Network-flow models: capacity, conservation, and maximum flow

How much flow can we send from a source to a sink through a network with capacity limits, and how can we show that this amount is the maximum?

> __Learning Objectives__
>
> At the end of this lecture, you should be able to:
>
> * __Formulate a maximum-flow problem:__ Represent a resource-allocation problem as a directed network with a source, sink, and edge capacities. Write the capacity and flow-conservation constraints, define the net flow leaving the source, and explain how maximizing this value relates to the application.
> * __Apply the Ford–Fulkerson method:__ Construct residual graphs and explain how backward edges allow earlier routing choices to be revised. Find the bottleneck capacity of an augmenting path, update the flow while preserving feasibility, and explain how Edmonds–Karp uses breadth-first search to choose the path.
> * __Verify a maximum flow:__ Check edge-capacity constraints, conservation at intermediate vertices, and the balance between net source outflow and net sink inflow. Use a source–sink cut to bound the flow value, and explain why the absence of an augmenting path establishes that the current feasible flow is maximum.

In this lecture, we will formulate and solve maximum-flow problems using residual graphs, then verify the result with a cut bound and interpret it through a worker–task assignment example.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines the lecture and data folder paths, and loads the course package and its dependencies.

Let's set up our code environment:


In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"))

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course package documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used here.

___

## Examples

We will use the following example to connect maximum flow to worker–task assignments:

> [▶ Maximum flow in a worker–task network](CHEME-5800-L5a-WorkedExample-MaximumFlow-Fall-2026.ipynb). How many assignments can the workers complete? We build the network from a workers–tasks edge list, compute maximum flows using depth-first and breadth-first searches to select augmenting paths, and check the results. The two searches can produce different assignments with the same maximum-flow value.

___


## Formulating a maximum-flow problem

How many tasks can a group of workers complete when each worker can accept only a limited number of assignments? We can represent assignments as flow through a network, with capacities limiting how much flow each connection can carry. In the worker–task example, one unit of flow represents one assignment. The same formulation applies to data transmission and material transport.

> __Flow network and constraints__
>
> Let $\mathcal{G}=(\mathcal{V},\mathcal{E})$ be a finite directed graph with vertex set $\mathcal{V}$ and edge set $\mathcal{E}$. The __source__ $s\in\mathcal{V}$ supplies flow, and the __sink__ $t\in\mathcal{V}$ receives it, with $s\neq t$. Each edge $(u,v)\in\mathcal{E}$ has a finite capacity $c(u,v)\geq0$. A __flow__ assigns a value $f(u,v)$ to each edge, in the same units as the capacity.
>
> __Capacity constraints.__ The flow on each original edge must remain between zero and its capacity:
> $$
> 0\leq f(u,v)\leq c(u,v),\qquad (u,v)\in\mathcal{E}.
> $$
>
> __Flow conservation.__ At every intermediate vertex $v\in\mathcal{V}\setminus\{s,t\}$, total inflow must equal total outflow:
> $$
> \sum_{u:(u,v)\in\mathcal{E}}f(u,v)=\sum_{w:(v,w)\in\mathcal{E}}f(v,w).
> $$
> Thus, an intermediate vertex neither creates nor consumes flow. A __feasible flow__ satisfies both the capacity and conservation constraints.

__Flow value.__ The value $|f|$ measures the net flow leaving the source. It is given by:
$$
|f|=
\underbrace{\sum_{v:(s,v)\in\mathcal{E}}f(s,v)}_{\text{flow leaving }s}
-\underbrace{\sum_{u:(u,s)\in\mathcal{E}}f(u,s)}_{\text{flow entering }s}.
$$
We subtract incoming flow because flow returning to the source does not add to the amount delivered. Conservation makes $|f|$ equal to the net flow entering the sink. In the worker–task network, no edges enter the source, so the second sum is zero.

The __maximum-flow problem__ seeks a feasible flow $f^*$ with the largest value:
$$
|f^*|=\max_{f\;\text{feasible}}|f|.
$$
For the worker–task network, this value gives the maximum number of assignments. We will compute it by starting with zero flow and increasing the flow while preserving both constraints.

___


## Ford-Fulkerson Method

The Ford–Fulkerson method begins with zero flow on every edge. We look for a path from the source to the sink that can carry flow and send as much along it as the edge capacities allow. We then look for a way to increase the total flow reaching the sink. Sometimes this requires moving flow from an earlier path to a different path.

The residual graph records the changes we can make to the current flow: where we can add more flow and where we can remove flow that was assigned earlier. Ford–Fulkerson is called a _method_ because it does not prescribe how to choose the paths.

Let's first construct the residual graph, then use it to understand how an augmentation changes the flow.

### What is a Residual Graph?

Suppose an edge from $u$ to $v$ has capacity $c(u,v)=5$ and currently carries flow $f(u,v)=3$. We can send two more units along this edge. We can also reduce its current flow by as many as three units if a different route would allow more flow to reach the sink. The residual graph represents these two possibilities with a forward edge of capacity two and a backward edge of capacity three.

For the formulas below, assume that the original network has no self-loops and no pair of edges pointing in opposite directions between the same two vertices. The worker–task network satisfies these assumptions. We continue to use $f(u,v)$ for the nonnegative flow on an original edge.

> __Residual Graph__
>
> For a feasible flow $f$ on the directed network $\mathcal{G}=(\mathcal{V},\mathcal{E})$, the __residual capacity__ describes how much flow we can add in a forward direction or cancel in a backward direction. Under the assumptions above, it is given by:
> $$
> \texttt{residual}(u,v)=
> \begin{cases}
> c(u,v)-f(u,v), & \text{if }(u,v)\in\mathcal{E},\\
> f(v,u), & \text{if }(v,u)\in\mathcal{E},\\
> 0, & \text{otherwise.}
> \end{cases}
> $$
> The __residual graph__ $\mathcal{G}_f=(\mathcal{V},\mathcal{E}_f)$ has the same vertices as the original network. Its edges are the ordered vertex pairs with positive residual capacity:
> $$
> \mathcal{E}_f=\left\{(u,v)\in\mathcal{V}\times\mathcal{V}:\texttt{residual}(u,v)>0\right\}.
> $$
> Thus, the residual graph can contain a backward edge even when that edge is absent from the original network. Using it cancels part of an earlier routing choice.

The backward edge is a bookkeeping device: it allows us to reduce an existing nonnegative edge flow. It does not require a physical connection that carries flow in the reverse direction.

### How do we augment the flow?

An __augmenting path__ $P$ is a directed path from $s$ to $t$ in the residual graph. Every edge on this path has positive residual capacity. Let $\Delta$ denote the amount by which we increase the source-to-sink flow. The largest allowable increase along the chosen path is set by its smallest residual capacity:
$$
\Delta=\min_{(u,v)\in P}\texttt{residual}(u,v).
$$
This is the path's __bottleneck capacity__. On a forward edge, it limits how much flow we can add before reaching the capacity. On a backward edge, it limits how much existing flow we can cancel before reaching zero.

__A reverse edge revises an earlier route.__ Consider a network with four vertices $s$, $a$, $b$, and $t$, and five original edges: $(s,a)$, $(s,b)$, $(a,b)$, $(a,t)$, and $(b,t)$. Every edge has capacity one. Suppose our first augmentation sends one unit along $s\to a\to b\to t$. This saturates three edges, including $(a,b)$, but the residual graph still contains the path $s\to b\to a\to t$.

![Three stages of a unit-capacity network: the first route sends one unit through a to b; a residual path uses the reverse edge from b to a; the revised flow sends two units along separate routes.](figs/Fig-L5a-ResidualRerouting.svg)

Each edge on the residual path has capacity one, so the second augmentation has $\Delta=1$. Traversing $b\to a$ cancels the earlier flow on $(a,b)$, while traversing $s\to b$ and $a\to t$ adds one unit to each of those original edges. The revised flow uses the two routes $s\to a\to t$ and $s\to b\to t$. Its value is two, and every intermediate vertex still conserves flow.

__Ford–Fulkerson procedure.__ Initialize $f(u,v)=0$ for every original edge $(u,v)\in\mathcal{E}$. Then repeat the following steps:

1. Construct the residual graph and find a directed path $P$ from $s$ to $t$. If no such path exists, stop and return the current flow.
2. Compute the bottleneck capacity $\Delta$ using the residual capacities along $P$.
3. For each residual edge $(u,v)$ on $P$, update the corresponding original edge:
   * If $(u,v)\in\mathcal{E}$, increase $f(u,v)$ by $\Delta$.
   * Otherwise, $(v,u)\in\mathcal{E}$; decrease $f(v,u)$ by $\Delta$.
4. Recompute the residual capacities before searching for the next path.

Because every edge on the path uses the same $\Delta$, these updates preserve flow conservation at intermediate vertices and increase the flow value by $\Delta$. The bottleneck condition keeps every original edge flow between zero and its capacity. We now need a rule for selecting the next augmenting path.

### How do we find augmenting paths?

Search the residual graph from $s$ to $t$, following only edges with positive residual capacity.

__Edmonds–Karp.__ This algorithm uses breadth-first search to choose a path with the fewest edges. When several paths tie, the neighbor order determines which path is found first. After each augmentation, update the residual capacities and search again.

With adjacency lists and exact arithmetic, the [worst-case running time](https://algs4.cs.princeton.edu/64maxflow/) is:
$$
\mathcal{O}\!\left(|\mathcal{V}|\,|\mathcal{E}|^2\right).
$$
The bound depends on the numbers of vertices and edges, not the capacity values.

__Depth-first search.__ This search also finds augmenting paths, but may use more edges. With nonnegative integer capacities and zero initial flow, each augmentation adds at least one unit. Since flow cannot exceed the total capacity leaving the source, the method terminates. The number of augmentations can depend on the capacity values. With irrational capacities, some sequences of path choices never terminate.

> __Example__
>
> [▶ Maximum flow in a worker–task network](CHEME-5800-L5a-WorkedExample-MaximumFlow-Fall-2026.ipynb). How many assignments can the workers complete? We build the network, compute the flow with both methods, and check the results. Both methods should give the same maximum-flow value, although their assignments may differ.

Next, we use a cut to show that the computed flow is maximum.

___


## Verifying feasibility and maximum flow

For any reported flow, we'll check the following conditions:

1. every original edge satisfies $0\leq f(u,v)\leq c(u,v)$;
2. every intermediate vertex conserves flow; and
3. the net flow leaving the source equals the net flow entering the sink.

These checks establish __feasibility__. To establish that the flow is __maximum__, we also need to show that no feasible flow can send more from the source to the sink. A cut through the network provides an upper bound that we can compare with the computed flow value.

> __An upper bound from a cut__
>
> A __source–sink cut__ partitions the vertices into two disjoint sets $S$ and $T$, with $s\in S$, $t\in T$, and $S\cup T=\mathcal{V}$. Its __capacity__ $c(S,T)$ is the sum of the original edge capacities directed from $S$ to $T$:
> $$
> c(S,T)=\sum_{\substack{(u,v)\in\mathcal{E}\\u\in S,\;v\in T}}c(u,v).
> $$
> Adding the net outflows of all vertices in $S$ cancels flows on edges within $S$. Conservation makes each intermediate vertex's contribution zero, so the sum equals the source's net outflow. This gives the cut balance:
> $$
> \begin{aligned}
> |f|&=\sum_{\substack{(u,v)\in\mathcal{E}\\u\in S,\;v\in T}}f(u,v)
> -\sum_{\substack{(u,v)\in\mathcal{E}\\u\in T,\;v\in S}}f(u,v)\\
> &\leq\sum_{\substack{(u,v)\in\mathcal{E}\\u\in S,\;v\in T}}c(u,v)=c(S,T).
> \end{aligned}
> $$
> The inequality follows because flow from $S$ to $T$ cannot exceed its edge capacities, and flow from $T$ to $S$ is nonnegative. Thus, every source–sink cut gives an upper bound on the value of any feasible flow.

__How does this explain the stopping condition?__ When no augmenting path remains, let $S$ contain the vertices reachable from $s$ in the residual graph, and let $T=\mathcal{V}\setminus S$. Since $t$ is unreachable, this partition is a source–sink cut.

<p align="center">
<img src="figs/Fig-Cut-Optimality/Fig-Cut-Optimality.svg" width="480" style="display:block; margin:0 auto;" alt="An illustrative five-vertex network with S containing s and a, saturated edges from a to b and c, and zero return flow from b to s. The feasible flow and cut capacity both equal two.">
</p>

__Illustrating the stopping condition.__ The two edges leaving $S$ are full, and the return edge carries no flow. Both the flow value and cut capacity are two.

An edge from $S$ to $T$ must be full; otherwise, the residual search could reach $T$. An edge from $T$ to $S$ must carry no flow, since any flow would create a backward residual edge into $T$. The cut balance then gives:
$$
|f|=c(S,T).
$$
The flow reaches the cut bound, so it is maximum. This is the [maximum-flow/minimum-cut theorem](https://courses.csail.mit.edu/6.854/21/Notes/n06-flow.html): the maximum flow value equals the smallest source–sink cut capacity.

In the worker–task network, the three source-to-worker edges have a total capacity of three assignments. The worked example reaches this bound, proving that the maximum is three assignments. In L5b, we examine how changing capacities affects this limit.

___


## Summary

In this lecture, we formulated maximum-flow problems, used residual graphs to increase flow, and proved that a feasible flow is maximum when its value equals a cut capacity.

> __Key Takeaways:__
>
> * __Capacity and conservation:__ We formulated maximum flow as the problem of maximizing net source outflow subject to edge capacities and conservation at intermediate vertices. In the worker–task network, one unit of flow represents one assignment, so the flow value tells us how many assignments the workers can complete.
>
> * __Augmentation with residual graphs:__ We used residual graphs to find ways to increase the flow, including canceling earlier routing choices through backward edges. The bottleneck capacity set the increase along each path while preserving feasibility. We compared depth-first search with Edmonds–Karp, which uses breadth-first search to choose a path with the fewest edges.
>
> * __Proof of maximum flow:__ We distinguished feasibility checks from a proof that the flow is maximum. When no augmenting path remained, the vertices reachable from the source defined a cut whose capacity equaled the flow value. Since every feasible flow is bounded by this capacity, the result was maximum. For the default worker–task network, a feasible flow and cut capacity of three established a maximum of three assignments.

In [the L5b lab](../L5b/CHEME-5800-L5b-Lab-MaximumFlowSensitivity-Fall-2026.ipynb), we will change edge capacities and examine how those changes affect the maximum number of assignments.
